## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [1]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


True

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [2]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")

In [3]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about robots")


In [4]:
# Here is the final output

print(result.final_output)

Why did the robot go on a diet?

Because it had too many bytes.


In [5]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about robots', 'role': 'user'},
 {'id': 'msg_0324978439a799cf006a525b08c2e88195b3a891f1f492ea54',
  'content': [{'annotations': [],
    'text': 'Why did the robot go on a diet?\n\nBecause it had too many bytes.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

## Adding Observability with a trace

In [7]:
with trace("robot joke"):
    result = await Runner.run(agent, "Tell a joke about robots")
print(result.final_output)

Why did the robot go on a diet?  
Because it had too many bytes.


## Now go and look at the trace

https://platform.openai.com/traces

In [8]:
# Streaming
with trace("5 jokes"):
    result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
    async for event in result.stream_events():
        if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
            print(event.data.delta, end="", flush=True)

Sure — here are 5 jokes about AI agents:

1. **Why did the AI agent bring a ladder to work?**  
   Because it heard the task was *multi-level*.

2. **My AI agent said it could “handle anything.”**  
   So I asked it to clean my room.  
   It responded: “I can optimize the clutter, but I cannot eliminate your lifestyle.”

3. **Why was the AI agent always calm?**  
   Because it never had emotions — just *very confident probabilities*.

4. **I asked my AI agent to be more creative.**  
   Now it gives me the same idea in seven different fonts.

5. **What’s an AI agent’s favorite kind of coffee?**  
   Decaf — because it already has too much *training data*.

If you want, I can also do:
- **more nerdy AI jokes**
- **short one-liners**
- **dad jokes about AI agents**

## Part 2: Adding a tool

In [9]:
# Getting the Telegram bot token and chat ID from environment variables
# You can also replace these with your actual values directly

TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN", "your_bot_token_here")
TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID", "your_chat_id_here")

### verify TELEGRAM token and chat_id are there

if TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID:
    print("TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID found")
else:
    print("TELEGRAM_BOT_TOKEN or TELEGRAM_CHAT_ID not found")

def send_telegram_message(text):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {"chat_id": TELEGRAM_CHAT_ID, "text": text}
    print(f"Sending message to Telegram: {text}")
    print(f"TELEGRAM_BOT_TOKEN: {TELEGRAM_BOT_TOKEN}")
    print(f"TELEGRAM_CHAT_ID: {TELEGRAM_CHAT_ID}")
    response = requests.post(url, data=payload)

    if response.status_code == 200:
        # print("Message sent successfully!")
        return {"status": "success", "message": text}
    else:
        # print(f"Failed to send message. Status code: {response.status_code}")
        # print(response.text)
        return {"status": "error", "message": response.text}

# send a test messagex
send_telegram_message("Test message Telegram")

TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID found
Sending message to Telegram: Test message Telegram
TELEGRAM_BOT_TOKEN: 8293258048:AAHVjgD8DLdWM7eG7dfgYqF8mlfOkd4QakM
TELEGRAM_CHAT_ID: 8058969323


{'status': 'success', 'message': 'Test message Telegram'}

In [14]:
send_telegram_message

<function __main__.send_telegram_message(text)>

In [15]:
# Now this:

@function_tool
def send_telegram_message_tool(text: str) -> str:
    """Use this when you want to send a message to the user as a push notification via Telegram"""
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {"chat_id": TELEGRAM_CHAT_ID, "text": text}
    print(f"Sending message to Telegram: {text}")
    
    response = requests.post(url, data=payload)

    if response.status_code == 200:
        # print("Message sent successfully!")
        return {"status": "success", "message": text}
    else:
        # print(f"Failed to send message. Status code: {response.status_code}")
        # print(response.text)
        return {"status": "error", "message": response.text}



In [20]:
send_telegram_message_tool

FunctionTool(name='send_telegram_message_tool', description='Use this when you want to send a message to the user as a push notification via Telegram', params_json_schema={'properties': {'text': {'title': 'Text', 'type': 'string'}}, 'required': ['text'], 'title': 'send_telegram_message_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x111bfa240>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [21]:
send_telegram_message_tool.description

'Use this when you want to send a message to the user as a push notification via Telegram'

In [ ]:

robo_joke_pusher = Agent(name="robo joke pusher", model="gpt-5.5", instructions="You are a funny bot. You can send robot jokes to users via Telegram messages. .", tools=[send_telegram_message_tool])

In [26]:
with trace("complex robo joke pusher"):
    result = await Runner.run(robo_joke_pusher, "The user needs a funny joke about robots.")

print(result.final_output)


Sending message to Telegram: I asked my robot vacuum why it looked so down. It said, “Life sucks… but at least I’m good at it.”
Sent the best robot joke via Telegram.


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [ ]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

## Memory approach 1 - just manually pass in the list of dicts

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

In [ ]:
response.to_input_list()

In [ ]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

In [ ]:
response = await Runner.run(agent, next_input)
print(response.final_output)

## Another approach - use OpenAI Agents SDK built in SQLLite session

In [ ]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>